In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../../.env")
pd.set_option("display.max_columns", None)

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
print("Connexion prête ✅")


Connexion prête ✅


In [2]:
# Pour la modélisation on prend un échantillon plus gros que l'EDA,
# mais pas les 5,6M (trop lourd pour Random Forest / XGBoost sur un PC portable).
# RAND(42) = graine fixe -> échantillon REPRODUCTIBLE (même tirage à chaque exécution).
query = "SELECT * FROM transactions WHERE RAND(42) < 0.1;"
df = pd.read_sql(query, con=engine)

print(f"Données chargées : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head()


Données chargées : 562,876 lignes × 23 colonnes


,Date mutation,Nature mutation,Valeur fonciere,Code postal,Commune,Code departement,Code commune,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain,prix_m2,annee,mois,trimestre,code_commune_geo,nom_commune_geo,codeDepartement,codeRegion,population_geo,longitude,latitude,revenu_median
0,2021-01-08,Vente,185000.0,1960.0,PERONNAS,01,289,Maison,100.0,4.0,703.0,1850.000000,2021,1,1,01289,Péronnas,01,84.0,6444.0,5.2195,46.1693,21929.1
1,2021-01-04,Vente,143000.0,1960.0,PERONNAS,01,289,Appartement,106.0,5.0,NaN,1349.056604,2021,1,1,01289,Péronnas,01,84.0,6444.0,5.2195,46.1693,21929.1
2,2021-01-07,Vente,72000.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,61.0,3.0,NaN,1180.327869,2021,1,1,01053,Bourg-en-Bresse,01,84.0,42372.0,5.2469,46.2027,18179.5
3,2021-01-21,Vente,100000.0,1370.0,SAINT-ETIENNE-DU-BOIS,01,350,Appartement,58.0,3.0,949.0,1724.137931,2021,1,1,01350,Saint-Étienne-du-Bois,01,84.0,2584.0,5.2793,46.2755,21421.0
4,2021-01-12,Vente,172000.0,1340.0,JAYAT,01,196,Maison,38.0,2.0,683.0,4526.315789,2021,1,1,01196,Jayat,01,84.0,1261.0,5.1117,46.3698,21040.4


In [3]:
# Nature mutation : on a filtré "Vente" à l'ETL, donc elle ne contient qu'UNE valeur
print(df["Nature mutation"].value_counts())


Nature mutation
Vente    562876
Name: count, dtype: int64


In [4]:
cols_a_supprimer = [
    # --- Fuite de données (data leakage) ---
    "prix_m2",              # = Valeur fonciere / Surface -> contient la réponse !

    # --- Colonne constante (aucune information) ---
    "Nature mutation",      # toujours "Vente"

    # --- Redondances de localisation (on garde Code departement + lat/lon + revenu) ---
    "Code postal",          # localisation redondante, trop de catégories
    "Commune",              # nom en texte, redondant avec les codes
    "Code commune",         # code local, redondant avec code_commune_geo
    "nom_commune_geo",      # texte, redondant
    "codeDepartement",      # doublon de "Code departement" (issu de l'API geo)

    # --- Redondances temporelles ---
    "Date mutation",        # déjà résumée par annee / mois
    "trimestre",            # redondant avec mois
]

df = df.drop(columns=cols_a_supprimer)

print(f"Colonnes restantes : {df.shape[1]}")
print(list(df.columns))


Colonnes restantes : 14
['Valeur fonciere', 'Code departement', 'Type local', 'Surface reelle bati', 'Nombre pieces principales', 'Surface terrain', 'annee', 'mois', 'code_commune_geo', 'codeRegion', 'population_geo', 'longitude', 'latitude', 'revenu_median']


In [5]:
na = df.isna().sum()
print(na[na > 0])
print(f"\nTotal lignes : {len(df):,}")


Surface terrain    214326
dtype: int64

Total lignes : 562,876


In [6]:
# Est-ce que les manquants de Surface terrain correspondent bien aux appartements ?
pd.crosstab(
    df["Type local"],
    df["Surface terrain"].isna(),
    rownames=["Type de bien"],
    colnames=["Surface terrain manquante ?"]
)


Surface terrain manquante ?,False,True
Type de bien,,
Appartement,37754,200875
Maison,310796,13451


In [7]:

# Vérification : plus aucun manquant
print("Manquants restants :", df.isna().sum().sum())


Manquants restants : 214326


**Traitement `Surface terrain` :**
- Appartements manquants → 0 (pas de terrain : correct).
- Maisons manquantes (~2,4 %) → 0 par simplification (valeur réellement inconnue).
  Approximation acceptable car modèles à base d'arbres robustes. Limite assumée.


In [8]:
from sklearn.model_selection import train_test_split

# X = toutes les features ; y = la cible
X = df.drop(columns=["Valeur fonciere"])
y = df["Valeur fonciere"]

# 80% entraînement / 20% test, tirage reproductible
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")


X_train : (450300, 13)
X_test  : (112576, 13)
y_train : (450300,)
y_test  : (112576,)


In [9]:
# Médiane du terrain des MAISONS, calculée sur le TRAIN uniquement (anti-fuite)
median_terrain_maison = X_train.loc[
    X_train["Type local"] == "Maison", "Surface terrain"
].median()

print(f"Médiane terrain des maisons (train) : {median_terrain_maison:.0f} m²")

def imputer_terrain(X, median_maison):
    X = X.copy()
    # Appartement manquant -> 0 (pas de terrain)
    mask_appart = (X["Type local"] == "Appartement") & (X["Surface terrain"].isna())
    X.loc[mask_appart, "Surface terrain"] = 0
    # Maison manquante -> médiane des maisons du train (valeur inconnue mais plausible)
    mask_maison = (X["Type local"] == "Maison") & (X["Surface terrain"].isna())
    X.loc[mask_maison, "Surface terrain"] = median_maison
    return X

# On applique la MÊME médiane (celle du train) aux deux jeux
X_train = imputer_terrain(X_train, median_terrain_maison)
X_test  = imputer_terrain(X_test,  median_terrain_maison)

print("Manquants train :", X_train.isna().sum().sum())
print("Manquants test  :", X_test.isna().sum().sum())


Médiane terrain des maisons (train) : 500 m²
Manquants train : 0
Manquants test  : 0


L'imputation par une statistique — médiane, moyenne — doit toujours être calculée sur le train et appliquée au test, pour éviter la fuite de données

In [10]:
from pathlib import Path

# Dossier dédié aux données prêtes pour le ML
out = Path("../../data/processed/ml")
out.mkdir(parents=True, exist_ok=True)

X_train.to_parquet(out / "X_train.parquet")
X_test.to_parquet(out / "X_test.parquet")
y_train.to_frame(name="Valeur fonciere").to_parquet(out / "y_train.parquet")
y_test.to_frame(name="Valeur fonciere").to_parquet(out / "y_test.parquet")

print("Jeux sauvegardés dans data/processed/ml/ ✅")


Jeux sauvegardés dans data/processed/ml/ ✅


In [11]:
X_train_check = pd.read_parquet(out / "X_train.parquet")
print("Relecture OK :", X_train_check.shape)
print(X_train_check.dtypes)


Relecture OK : (450300, 13)
Code departement              object
Type local                    object
Surface reelle bati          float64
Nombre pieces principales    float64
Surface terrain              float64
annee                          int64
mois                           int64
code_commune_geo              object
codeRegion                   float64
population_geo               float64
longitude                    float64
latitude                     float64
revenu_median                float64
dtype: object


## 🧾 Conclusions du préprocessing

- Échantillon reproductible de ~563k lignes (RAND(42)), stratégie : dev sur échantillon, modèle final sur 5,6M.
- 23 → 12 features : suppression de la fuite (`prix_m2`), colonne constante, redondances géo/temporelles.
- `Surface terrain` : appartements → 0, maisons manquantes → médiane des maisons du **train** (anti-fuite).
- Split 80/20 (`random_state=42`) fait AVANT toute transformation apprise.
- Jeux sauvegardés dans `data/processed/ml/`.

**Reste à faire (notebooks 03/04) :** feature engineering, encodage catégoriel (sur train), normalisation si besoin, `log1p` sur la cible.
